# 3. Database Security — SQL Entra Auth, TDE, Dynamic Masking, Always Encrypted

## Azure SQL security layers

```
┌── Network ──────────────────────────────────────────────┐
│  Firewall rules, Private Endpoints, VNet service rules  │
│ ┌── Authentication ──────────────────────────────────┐  │
│ │  Entra ID (preferred), SQL auth (legacy)           │  │
│ │ ┌── Authorization ──────────────────────────────┐  │  │
│ │ │  Database roles, row-level security            │  │  │
│ │ │ ┌── Data protection ───────────────────────┐  │  │  │
│ │ │ │  TDE, Always Encrypted, dynamic masking  │  │  │  │
│ │ │ └─────────────────────────────────────────┘  │  │  │
│ │ └──────────────────────────────────────────────┘  │  │
│ └──────────────────────────────────────────────────┘  │
│ ┌── Monitoring ─────────────────────────────────────┐  │
│ │  Auditing, threat detection (Defender for SQL)     │  │
│ └──────────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────────┘
```

## Entra ID authentication for SQL

```bash
# Set Entra admin on SQL Server
az sql server ad-admin create -g rg-prod -s sql-prod \
  --display-name 'DBA Team' --object-id <group-object-id>

# Connect using Entra ID (from az CLI or app)
# In connection string: Authentication=Active Directory Default
```

**Exam tip**: set an Entra group as SQL admin (not an individual). Use managed identities for application access.

In [ ]:
# Simulate dynamic data masking
MASKING_RULES = [
    {'column': 'email',       'function': 'email',   'description': 'aXXX@XXXX.com'},
    {'column': 'credit_card', 'function': 'partial', 'description': 'XXXX-XXXX-XXXX-1234', 'prefix': 0, 'suffix': 4, 'padding': 'XXXX-XXXX-XXXX-'},
    {'column': 'ssn',         'function': 'default', 'description': 'XXXX'},
    {'column': 'phone',       'function': 'partial', 'description': 'XXX-XXX-5678', 'prefix': 0, 'suffix': 4, 'padding': 'XXX-XXX-'},
    {'column': 'salary',      'function': 'random',  'description': 'Random value in range', 'min': 30000, 'max': 200000},
]

SAMPLE_DATA = [
    {'name': 'Alice Johnson', 'email': 'alice.johnson@contoso.com', 'credit_card': '4111-1111-1111-1234', 'ssn': '123-45-6789', 'phone': '555-123-5678', 'salary': 145000},
    {'name': 'Bob Smith',     'email': 'bob.smith@contoso.com',     'credit_card': '5500-0000-0000-5678', 'ssn': '987-65-4321', 'phone': '555-456-9012', 'salary': 98000},
]

import random

def apply_mask(value, rule):
    fn = rule['function']
    if fn == 'default':
        return 'XXXX'
    elif fn == 'email':
        parts = str(value).split('@')
        return f'{parts[0][0]}XXX@XXXX.com'
    elif fn == 'partial':
        s = str(value)
        return rule['padding'] + s[-rule['suffix']:]
    elif fn == 'random':
        return random.randint(rule['min'], rule['max'])
    return value

def query_as(role: str, data: list, mask_rules: list) -> list:
    if role == 'db_owner':  # unmasked
        return data
    masked = []
    rule_map = {r['column']: r for r in mask_rules}
    for row in data:
        masked_row = {}
        for col, val in row.items():
            if col in rule_map:
                masked_row[col] = apply_mask(val, rule_map[col])
            else:
                masked_row[col] = val
        masked.append(masked_row)
    return masked

print('=== Dynamic Data Masking ===\n')
print('--- View as db_owner (full access) ---')
for row in query_as('db_owner', SAMPLE_DATA, MASKING_RULES):
    print(f'  {row}')

print('\n--- View as app_user (masked) ---')
for row in query_as('app_user', SAMPLE_DATA, MASKING_RULES):
    print(f'  {row}')

print('\n💡 Masking is applied at query time. db_owner and users with UNMASK permission see real data.')
print('⚠️ Masking is NOT encryption — a determined user can infer masked values with queries.')

## Transparent Data Encryption (TDE)

Encrypts the database **at rest** (data files, log files, backups). Enabled by default for Azure SQL.

| TDE Key Management | Description |
|-------------------|-------------|
| **Service-managed** | Microsoft manages the key (default) |
| **Customer-managed (BYOK)** | Your key in Key Vault |

```bash
# Check TDE status
az sql db tde show -g rg-prod -s sql-prod -d mydb

# Configure TDE with customer-managed key
az sql server key create -g rg-prod -s sql-prod \
  --kid https://my-kv.vault.azure.net/keys/sql-tde-key/version

az sql server tde-key set -g rg-prod -s sql-prod \
  --server-key-type AzureKeyVault \
  --kid https://my-kv.vault.azure.net/keys/sql-tde-key/version
```

## Always Encrypted

Unlike TDE (which the database engine can decrypt), Always Encrypted keeps data encrypted **even from the database engine and DBAs**. Only the application with the Column Master Key (CMK) can decrypt.

| | TDE | Always Encrypted |
|-|-----|------------------|
| Protects from | Stolen disk/backup | DBAs, cloud provider, SQL injection |
| Encryption scope | Entire database | Individual columns |
| Key holder | SQL Server / Azure | Client application only |
| Performance impact | Minimal | Higher (client-side encrypt/decrypt) |
| Query support | Full | Limited (equality only for deterministic) |

**When to recommend Always Encrypted**: sensitive columns (SSN, credit cards) where even DBAs shouldn't see plaintext.

## Auditing

```bash
# Enable auditing to Log Analytics
az sql server audit-policy update -g rg-prod -n sql-prod \
  --state Enabled \
  --log-analytics-target-state Enabled \
  --log-analytics-workspace-resource-id /subscriptions/.../workspaces/la-security
```

Auditing records: logins, queries, schema changes, data changes. Store in:
- **Storage account** (long-term retention, cheapest)
- **Log Analytics** (query with KQL, integrate with Sentinel)
- **Event Hub** (stream to external SIEM)

---
## Summary

| Feature | Key exam fact |
|---------|---------------|
| **Entra auth for SQL** | Set Entra group as admin. Use managed identity for apps. |
| **Dynamic masking** | Query-time masking. Not encryption. UNMASK permission bypasses it. |
| **TDE** | At-rest encryption. On by default. BYOK via Key Vault. |
| **Always Encrypted** | Client-side encryption. Even DBAs can't read plaintext. |
| **Auditing** | Log to Storage, Log Analytics, or Event Hub. |

**Next lab**: [04 — Defender for Cloud & Sentinel](../../04-defender-and-sentinel/)